In [3]:
import os
from IPython.display import display
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'
from onekey_algo import OnekeyDS as okds
from onekey_algo import get_param_in_cwd

os.makedirs('img', exist_ok=True)
os.makedirs('results', exist_ok=True)

task_type = ''
mydir = get_param_in_cwd('radio_dir') or okds.ct

group_info = get_param_in_cwd('dataset_column') or 'group'
labelf = get_param_in_cwd('label_file') or os.path.join(mydir, 'label.csv')

labels = [get_param_in_cwd('task_column') or 'label']

In [4]:
from pathlib import Path

In [1]:
import warnings
import pandas as pd
 
warnings.filterwarnings("ignore")

from onekey_algo.custom.components.Radiology import ConventionalRadiomics

if os.path.exists('features/rad_features.csv'):
    rad_data = pd.read_csv('features/rad_features.csv', header=0)
else:

    param_file = get_param_in_cwd('extractor_settings')
    if param_file and not os.path.exists(param_file):
        print(f"{param_file}！！！！")
    radiomics = ConventionalRadiomics(param_file, correctMask=True)
    radiomics.extract(images, masks, workers=1)
    rad_data = radiomics.get_label_data_frame(label=1)
    rad_data.columns = [c.translate(str.maketrans({'-': '_'})) for c in rad_data.columns]
    rad_data.to_csv('features/rad_features.csv', header=True, index=False)
rad_data.head()

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns
sorted_counts = pd.DataFrame([c.split('_')[-3] for c in rad_data.columns if c !='ID']).value_counts()
sorted_counts = pd.DataFrame(sorted_counts, columns=['count']).reset_index()
sorted_counts = sorted_counts.sort_values(0)
sorted_counts.columns = ['feature_group', 'count']
display(sorted_counts)

In [3]:
label_data = pd.read_csv(labelf)
label_data['ID'] = label_data['ID'].map(lambda x: f"{x}.nii.gz" if not (f"{x}".endswith('.nii.gz') or  f"{x}".endswith('.nii')) else x)
label_data = label_data[['ID', 'group'] + labels]
label_data.head()

In [4]:
from onekey_algo.custom.utils import print_join_info

print_join_info(rad_data, label_data)
combined_data = pd.merge(rad_data, label_data, on=['ID'], how='inner')
ids = combined_data['ID']
combined_data = combined_data.drop(['ID'], axis=1)
print(combined_data[labels].value_counts())
combined_data

In [5]:
combined_data.describe()

In [6]:
from onekey_algo.custom.components.comp1 import normalize_df
data = normalize_df(combined_data, not_norm=labels, group=group_info)
data = data.dropna(axis=1)
data.describe()

In [7]:
import seaborn as sns
from onekey_algo.custom.components.stats import clinic_stats

stats = clinic_stats(data[data['group'] == 'train'], stats_columns=list(data.columns[0:-2]), label_column=labels[0], 
                     continuous_columns=list(data.columns[0:-2]))
stats

In [14]:
import matplotlib.pyplot as plt

def map2float(x):
    try:
        return float(str(x)[1:])
    except:
        return 1

stats[['pvalue']] = stats[['pvalue']].applymap(map2float)
stats[['group']] = stats[['feature_name']].applymap(lambda x: x.split('_')[-2])
stats = stats[['feature_name', 'pvalue', 'group']]

In [8]:
pvalue = 0.05
sel_feature = list(stats[stats['pvalue'] < pvalue]['feature_name']) + labels + [group_info]
data = data[sel_feature]
data

In [17]:
pearson_corr = data[data['group'] == 'train'][[c for c in data.columns if c not in labels]].corr('pearson')

In [ ]:
sel_data = data[sel_feature]
sel_data

In [11]:
import numpy as np
import onekey_algo.custom.components as okcomp

n_classes = 2
train_data = sel_data[(sel_data[group_info] == 'train')]
train_ids = ids[train_data.index]
train_data = train_data.reset_index()
train_data = train_data.drop('index', axis=1)
y_data = train_data[labels]
X_data = train_data.drop(labels + [group_info], axis=1)

val_data = sel_data[sel_data[group_info] == 'val']
val_ids = ids[val_data.index]
val_data = val_data.reset_index()
val_data = val_data.drop('index', axis=1)
y_val_data = val_data[labels]
X_val_data = val_data.drop(labels + [group_info], axis=1)

test_data = sel_data[sel_data[group_info] == 'test']
test_ids = ids[test_data.index]
test_data = test_data.reset_index()
test_data = test_data.drop('index', axis=1)
y_test_data = test_data[labels]
X_test_data = test_data.drop(labels + [group_info], axis=1)

y_all_data = sel_data[labels]
X_all_data = sel_data.drop(labels + [group_info], axis=1)

column_names = X_data.columns
print(f"{X_data.shape}, {X_val_data.shape}, {X_test_data.shape}")

In [14]:
alpha = okcomp.comp1.lasso_cv_coefs(X_data, y_data, column_names=None)
plt.savefig(f'img/{task_type}_feature_lasso.png', bbox_inches = 'tight')

In [15]:
okcomp.comp1.lasso_cv_efficiency(X_data, y_data, points=50)
plt.savefig(f'img/{task_type}_feature_mse.svg', bbox_inches = 'tight')

In [16]:
from sklearn import linear_model

model = linear_model.Lasso(alpha=alpha)
model.fit(X_data, y_data[labels])

In [18]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

COEF_THRESHOLD = 0  
scores = []
selected_features = []

for idx, label in enumerate(labels):
    feat_coef = [(feat_name, coef) for feat_name, coef in zip(column_names, model.coef_) 
                 if COEF_THRESHOLD is None or abs(coef) > COEF_THRESHOLD]
    selected_features.append([feat for feat, _ in feat_coef])
    formula = ' '.join([f"{coef:+.6f} * {feat_name}" for feat_name, coef in feat_coef])
    score = f"{label} = {model.intercept_[idx]} {'+' if formula[0] != '-' else ''} {formula}"
    scores.append(score)
    
    feat_coef = sorted(feat_coef, key=lambda x: x[1])
    feat_coef_df = pd.DataFrame(feat_coef, columns=['feature_name', 'Coefficients'])
    

    fig, ax = plt.subplots(figsize=(10, 6))
    y_pos = np.arange(len(feat_coef_df))
    
    for i, (feat, coef) in enumerate(feat_coef_df.values):
      
        ax.hlines(y=i, xmin=0, xmax=coef, color=(160/255, 70/255, 90/255), linewidth=3)
        
        markersize = 5 + abs(coef) * 50
        ax.plot(coef, i, "o", color=(120/255, 0, 50/255), markersize=markersize)  
    
    ax.set_yticks(y_pos)
    ax.set_yticklabels(feat_coef_df['feature_name'])
    ax.set_xlabel('Coefficients')
    ax.set_ylabel('Feature Names')
    ax.set_title(f'')
    
    plt.tight_layout()
    plt.savefig(f'img/{label}_{task_type}Rad_feature_weights_lollipop.svg', bbox_inches='tight')

print('\n'.join(scores))

In [19]:
X_data = X_data[selected_features[0]]
X_val_data = X_val_data[selected_features[0]]
X_test_data = X_test_data[selected_features[0]]
X_data.columns

In [33]:
model_names = get_param_in_cwd('ml_models')
models = okcomp.comp1.create_clf_model_none_overfit(model_names)
model_names = list(models.keys())

In [20]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, roc_auc_score

for l in labels:
    results = okcomp.comp1.get_bst_split(X_data, y_data[[l]], models, test_size=0.2, metric_fn=roc_auc_score, n_trails=5, cv=False, 
                                         random_state=0)
    _, (X_train_sel, X_test_sel, y_train_sel, y_test_sel) = results['results'][results['max_idx']]
    X_train_sel, X_val_sel, X_test_sel, y_train_sel, y_val_sel, y_test_sel = X_data, X_val_data, X_test_data, y_data, y_val_data, y_test_data
    train_ids_sel, test_ids_sel = train_ids, test_ids
    trails, _ = zip(*results['results'])
    cv_results = pd.DataFrame(trails, columns=model_names)

    sns.boxplot(data=cv_results)
    plt.ylabel('AUC %')
    plt.xlabel('Model Nmae')
    # plt.xticks(rotation=90)
    # plt.ylim(0.5,)
    plt.savefig(f'img/{l}_{task_type}Rad_model_cv.svg', bbox_inches = 'tight')
    plt.show()

In [35]:
import joblib
from onekey_algo.custom.components.comp1 import plot_feature_importance, plot_learning_curve, smote_resample
targets = []
os.makedirs('models', exist_ok=True)
for l in labels:
    new_models = list(okcomp.comp1.create_clf_model_none_overfit(model_names).values())
    for mn, m in zip(model_names, new_models):
        X_train_smote, y_train_smote = X_train_sel, y_train_sel

        m.fit(X_train_smote, y_train_smote[l])

        joblib.dump(m, f'models/{task_type}_{mn}_{l}.pkl') 

    targets.append(new_models)

In [21]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import OneHotEncoder
from onekey_algo.custom.components.delong import calc_95_CI
from onekey_algo.custom.components.metrics import analysis_pred_binary

def calculate_accuracy_confidence_interval(y_true, y_pred, z=1.96):
    acc = accuracy_score(y_true, y_pred)
    n = len(y_true)
    variance = acc * (1 - acc)
    margin_error = z * np.sqrt(variance / n)
    ci_lower = acc - margin_error
    ci_upper = acc + margin_error
    return ci_lower, ci_upper

metric = []
pred_sel_idx = []
predictions = [[(model.predict(X_train_sel), model.predict(X_val_sel), model.predict(X_test_sel))  
                for model in target] for label, target in zip(labels, targets)]
pred_scores = [[(model.predict_proba(X_train_sel), model.predict_proba(X_val_sel), model.predict_proba(X_test_sel)) 
                for model in target] for label, target in zip(labels, targets)]

for label, prediction, scores in zip(labels, predictions, pred_scores):
    pred_sel_idx_label = []
    for mname, (train_pred, val_pred, test_pred), (train_score, val_score, test_score) in zip(model_names, prediction, scores):
        # Calculate train set metrics
        train_ci_acc = calculate_accuracy_confidence_interval(y_train_sel[label], train_pred)
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_train_sel[label], 
                                                                                          train_score[:, 1])
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        train_ci_acc_formatted = f"{train_ci_acc[0]:.4f} - {train_ci_acc[1]:.4f}"
        metric.append((mname, acc, train_ci_acc_formatted, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f"Train"))

        # Calculate validation set metrics
        val_ci_acc = calculate_accuracy_confidence_interval(y_val_sel[label], val_pred)
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_val_sel[label], 
                                                                                          val_score[:, 1])
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        val_ci_acc_formatted = f"{val_ci_acc[0]:.4f} - {val_ci_acc[1]:.4f}"
        metric.append((mname, acc, val_ci_acc_formatted, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f'Val'))

        # Calculate test set metrics
        test_ci_acc = calculate_accuracy_confidence_interval(y_test_sel[label], test_pred)
        acc, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres = analysis_pred_binary(y_test_sel[label], 
                                                                                          test_score[:, 1])
        ci = f"{ci[0]:.4f} - {ci[1]:.4f}"
        test_ci_acc_formatted = f"{test_ci_acc[0]:.4f} - {test_ci_acc[1]:.4f}"
        metric.append((mname, acc, test_ci_acc_formatted, auc, ci, tpr, tnr, ppv, npv, precision, recall, f1, thres, f'Test'))

        # Calculate thres corresponding sel idx
        pred_sel_idx_label.append(np.logical_or(test_score[:, 0] >= thres, test_score[:, 1] >= thres))

    pred_sel_idx.append(pred_sel_idx_label)
metric = pd.DataFrame(metric, index=None, columns=['Model Name', 'Acc', 'Acc 95%CI', 'AUC', 'AUC 95%CI', 
                                                   'Sen', 'Spe', 'PPV', 'NPV', 'Precision', 
                                                   'Recall', 'F1', 'Threshold', 'Cohort'])

metric

In [52]:
import os
import numpy as np

os.makedirs('results', exist_ok=True)
sel_model = sel_model

for idx, label in enumerate(labels):
    for sm in sel_model:
        if sm in model_names:
            sel_model_idx = model_names.index(sm)
            target = targets[idx][sel_model_idx]

            train_indexes = np.reshape(np.array(train_ids), (-1, 1)).astype(str)
            val_indexes = np.reshape(np.array(val_ids), (-1, 1)).astype(str)
            test_indexes = np.reshape(np.array(test_ids_sel), (-1, 1)).astype(str)
            y_train_pred_scores = target.predict_proba(X_train_sel)
            y_val_pred_scores = target.predict_proba(X_val_sel)
            y_test_pred_scores = target.predict_proba(X_test_sel)
            columns = ['ID'] + [f"{label}-{i}"for i in range(y_test_pred_scores.shape[1])]

            result_train = pd.DataFrame(np.concatenate([train_indexes, y_train_pred_scores], axis=1), columns=columns)
            result_train.to_csv(f'results/{task_type}_{sm}_train.csv', index=False)
            result_val = pd.DataFrame(np.concatenate([val_indexes, y_val_pred_scores], axis=1), columns=columns)
            result_val.to_csv(f'results/{task_type}_{sm}_val.csv', index=False)
            result_test = pd.DataFrame(np.concatenate([test_indexes, y_test_pred_scores], axis=1), columns=columns)
            result_test.to_csv(f'results/{task_type}_{sm}_test.csv', index=False)